# einops-reduce — ex10: weighted-reduce via einsum: uniform identity + class-balanced loss

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. Running the final beacon cell reports progress against the `Einops: Reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## reduce('mean') vs einsum-weighted reduce — quick refresher

`einops.reduce(x, 'b n -> b', 'mean')` is mathematically identical to a uniform-weighted einsum:

```python
w = t.full((N,), 1.0 / N)
einsum('b n, n -> b', x, w)
```

But `reduce` only supports uniform weights via its named ops (`'sum'`, `'mean'`, `'max'`, `'min'`, `'prod'`). When you need **non-uniform weights** (class-balanced loss, importance weighting, gated attention), drop down to einsum.

**This drill (ex10) vs ex1-9.** Earlier exercises stayed inside reduce's named ops (channel mean, max-keepdim, BN stats, argmax-via-reduce). ex10 establishes the einsum-equivalence numerically and then USES it for a non-uniform pattern — class-weighted loss averaging — that reduce cannot express.

### Exercise 10 — weighted-reduce via einsum: uniform identity + class-balanced loss

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the relationship between `einops.reduce('mean')` and a uniform-weight einsum, then apply a non-uniform weight vector to compute a class-balanced per-batch loss that `reduce` alone cannot express.
> Keywords: weighted-reduce, einsum-equivalence, class-balanced-loss
> ```

**KCs targeted:** `reduce-mean-as-uniform-einsum`, `reduce-cannot-express-weighted`

Implement `ex10_weighted_reduce(losses, class_ids, class_weights)`.

Walk three reductions on the same `(B, N)` tensor of per-sample losses, where `class_ids` (length `B`) names the class of each sample and `class_weights` (length `C`) gives the per-class weight:

1. **Uniform reduce.** `mean_reduce = reduce(losses, 'b n -> b', 'mean')` — the standard `(B,)` per-batch mean.
2. **Uniform einsum (equivalent).** `mean_einsum = einsum('b n, n -> b', losses, t.full((N,), 1/N))`. Assert this matches `mean_reduce` to within `1e-6`.
3. **Class-balanced batch loss (genuinely different).** For each sample `b`, look up its weight `w_b = class_weights[class_ids[b]]`. Compute the class-balanced batch loss as `t.einsum('b, b ->', mean_reduce, w_b_vec) / w_b_vec.sum()` — a scalar.

Return `(mean_reduce, mean_einsum, class_balanced_loss)`. Print all three so the caller sees the chain.

Inputs: `losses` `(B, N)` float; `class_ids` `(B,)` int64 in `[0, C)`; `class_weights` `(C,)` float.
Output: `(mean_reduce, mean_einsum, class_balanced_loss)`.

The visualization compares per-batch unweighted vs class-weighted contribution as a bar chart, highlighting how the class weighting amplifies or suppresses each batch sample.

In [ ]:
def ex10_weighted_reduce(
    losses: Tensor,
    class_ids: Tensor,
    class_weights: Tensor,
) -> tuple[Tensor, Tensor, Tensor]:
    """Uniform reduce == uniform einsum; then a class-weighted scalar."""
    raise NotImplementedError()


def _test_ex10():
    # Hand-checked case.
    losses = t.tensor([
        [1.0, 2.0, 3.0, 4.0],   # batch 0, class 1, mean=2.5
        [0.5, 0.5, 0.5, 0.5],   # batch 1, class 0, mean=0.5
        [4.0, 4.0, 4.0, 4.0],   # batch 2, class 1, mean=4.0
    ])
    class_ids = t.tensor([1, 0, 1], dtype=t.long)
    class_weights = t.tensor([1.0, 3.0])  # class 1 is 3x as important as class 0
    m_r, m_e, cb = ex10_weighted_reduce(losses, class_ids, class_weights)

    # Per-batch mean is straightforward.
    assert t.allclose(m_r, t.tensor([2.5, 0.5, 4.0])), f'mean_reduce wrong: {m_r}'
    # Uniform-einsum must match uniform-reduce.
    assert t.allclose(m_r, m_e, atol=1e-6), f'einsum disagrees with reduce: {m_e} vs {m_r}'

    # Class-balanced loss = (3*2.5 + 1*0.5 + 3*4.0) / (3+1+3) = (7.5+0.5+12)/7 = 20/7
    expected_cb = (3*2.5 + 1*0.5 + 3*4.0) / (3 + 1 + 3)
    assert abs(cb.item() - expected_cb) < 1e-5, f'cb wrong: got {cb.item()}, expected {expected_cb}'

    # Stress test: random data, B=16, N=20, C=4.
    rng = t.Generator().manual_seed(8)
    L = t.rand(16, 20, generator=rng)
    cids = t.randint(0, 4, (16,), generator=rng)
    cw = t.tensor([0.5, 1.0, 2.0, 4.0])
    m1, m2, c = ex10_weighted_reduce(L, cids, cw)
    assert m1.shape == (16,) and m2.shape == (16,)
    assert c.dim() == 0  # scalar
    assert t.allclose(m1, m2, atol=1e-5), 'einsum-uniform must equal reduce-mean'

    # --- Visualization: per-batch unweighted vs class-weighted contribution ---
    fig, ax = plt.subplots(figsize=(6, 3))
    B = m1.shape[0]
    ws = cw[cids]  # (B,)
    ax.bar(range(B), m1.numpy(), label='unweighted batch mean',
           color='steelblue', alpha=0.7)
    ax.bar(range(B), (m1 * ws / ws.sum()).numpy(),
           label='class-weighted contribution', color='coral', alpha=0.9)
    ax.axhline(c.item(), color='black', linestyle='--', label=f'class-balanced loss = {c.item():.3f}')
    ax.set_xlabel('batch index')
    ax.set_ylabel('value')
    ax.set_title('ex10 class-weighted reduce amplifies high-weight samples')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex10')
    print("ex10 ✓")

_test_ex10()

<details><summary>Solution</summary>

```python
def ex10_weighted_reduce(
    losses: Tensor,
    class_ids: Tensor,
    class_weights: Tensor,
) -> tuple[Tensor, Tensor, Tensor]:
    B, N = losses.shape

    # 1. uniform reduce
    mean_reduce = reduce(losses, 'b n -> b', 'mean')

    # 2. uniform einsum equivalent
    uniform_w = t.full((N,), 1.0 / N, dtype=losses.dtype)
    mean_einsum = t.einsum('b n, n -> b', losses, uniform_w)

    # 3. class-balanced batch loss
    w_b_vec = class_weights[class_ids]                              # (B,)
    class_balanced_loss = t.einsum('b, b ->', mean_reduce, w_b_vec) / w_b_vec.sum()

    print(f'  mean_reduce = {mean_reduce}')
    print(f'  mean_einsum = {mean_einsum}')
    print(f'  class_balanced_loss = {class_balanced_loss.item():.5f}')
    return mean_reduce, mean_einsum, class_balanced_loss
```

**Why reduce('mean') == einsum-with-1/N.** Both compute `sum_i x_bi · w_i` with `w_i = 1/N` for all `i`. The reduce-vs-einsum distinction is one of API style, not numerics. Verifying this equality is the easiest way to convince yourself that the einsum-weighted form is correct before you generalize to non-uniform weights.

**Why you can't do class-weighted reduce in pure einops.** `einops.reduce` named ops are uniform by construction — there's no kwarg for per-element weights. The moment you need `w_i ≠ 1/N`, you've left the reduce API and entered einsum territory. (Or you cheat with `t.sum(x * w_broadcast, dim=...)`, but that's just einsum with extra steps.)

**Class-balanced loss in practice.** Imbalanced training sets give the majority class a free win on plain `loss.mean()` — the majority pulls the gradient. Multiplying each sample's per-class weight (typically `1 / class_frequency`) before averaging makes the per-step gradient class-frequency-invariant. The pattern in this drill is exactly what HuggingFace's class-balanced loss helper computes under the hood.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()